# 01_data_audit

Research Project B data audit for BBL first-innings checkpoint prediction.

Goal of this notebook:
- load the final modelling dataset
- verify target and feature definitions
- identify leakage / redundant columns
- create a clean modelling table for checkpoint-specific experiments

In [15]:
# Step 1: setup and load data

from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Find project root whether notebook is run from repo root or notebooks/Project_B
cwd = Path.cwd()

if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
elif (cwd.parent.parent / "data").exists():
    PROJECT_ROOT = cwd.parent.parent
else:
    raise FileNotFoundError("Could not find project root containing the data/ folder.")

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

model_path = PROCESSED_DIR / "bbl_model_dataset_with_elo.csv"

df = pd.read_csv(model_path)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Loaded:", model_path)
print("Shape:", df.shape)

PROJECT_ROOT: /Users/adithyacpatil/Research Project A/cleaning/MDS-cricket-project-public
Loaded: /Users/adithyacpatil/Research Project A/cleaning/MDS-cricket-project-public/data/processed/bbl_model_dataset_with_elo.csv
Shape: (2404, 39)


## Step 2: basic dataset audit

In this step, we inspect:
- column names
- data types
- missing values
- checkpoint values and row counts

This gives us a clean baseline before checking the target and deciding what to drop.

In [16]:
# Step 2: basic dataset audit

print("Number of columns:", len(df.columns))
print("\nColumns:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:02d}. {col}")

print("\nData types:")
print(df.dtypes.sort_index())

print("\nMissing values per column:")
missing = df.isna().sum().sort_values(ascending=False)
print(missing[missing > 0])

print("\nCheckpoint values:")
print(sorted(df["checkpoint"].dropna().unique()))

print("\nRows per checkpoint:")
print(df["checkpoint"].value_counts().sort_index())

print("\nFirst 5 rows:")
display(df.head())

Number of columns: 39

Columns:
01. match_id
02. checkpoint
03. runs
04. wickets
05. overs_left
06. run_rate
07. runs_per_wicket
08. pressure_index
09. checkpoint_ratio
10. momentum_runs
11. momentum_wickets
12. momentum_runrate
13. momentum_score
14. venue
15. season
16. toss_winner
17. winner
18. batting_first_team
19. target_batfirst
20. par_runs_cp_final
21. exp_remain_final
22. par_gap_cp
23. exp_final_20
24. resource_frac
25. venue_mean_runs_cp
26. venue_mean_wkts_cp
27. season_mean_runs_cp
28. season_mean_wkts_cp
29. venue_mean_exp_final20
30. team1
31. team2
32. elo_team1_pre
33. elo_team2_pre
34. elo_diff_pre
35. elo_bat_first
36. elo_bowl_first
37. elo_diff
38. runs_x_wickets
39. r_pw_x_momentum

Data types:
batting_first_team         object
checkpoint                  int64
checkpoint_ratio          float64
elo_bat_first             float64
elo_bowl_first            float64
elo_diff                  float64
elo_diff_pre              float64
elo_team1_pre             float64


,match_id,checkpoint,runs,wickets,overs_left,run_rate,runs_per_wicket,pressure_index,checkpoint_ratio,momentum_runs,momentum_wickets,momentum_runrate,momentum_score,venue,season,toss_winner,winner,batting_first_team,target_batfirst,par_runs_cp_final,exp_remain_final,par_gap_cp,exp_final_20,resource_frac,venue_mean_runs_cp,venue_mean_wkts_cp,season_mean_runs_cp,season_mean_wkts_cp,venue_mean_exp_final20,team1,team2,elo_team1_pre,elo_team2_pre,elo_diff_pre,elo_bat_first,elo_bowl_first,elo_diff,runs_x_wickets,r_pw_x_momentum
0,524915,6,37,0,14,6.166667,37.000000,2.466667,0.30,33,0,6.6,6.6,"Sydney Cricket Ground, Sydney",2011,Brisbane Heat,Sydney Sixers,Brisbane Heat,0,53.710280,122.364486,-16.710280,159.364486,0.757142,45.982759,1.620690,46.333333,0.666667,158.975113,Sydney Sixers,Brisbane Heat,1500.0,1500.0,0.0,1500.0,1500.0,0.0,0,244.200000
1,524915,10,50,3,10,5.000000,12.500000,1.136364,0.50,22,3,4.4,-1.6,"Sydney Cricket Ground, Sydney",2011,Brisbane Heat,Sydney Sixers,Brisbane Heat,0,69.302013,84.684564,-19.302013,134.684564,0.523994,73.894737,2.649123,74.250000,1.666667,158.975113,Sydney Sixers,Brisbane Heat,1500.0,1500.0,0.0,1500.0,1500.0,0.0,150,-20.000000
2,524915,15,104,5,5,6.933333,17.333333,2.888889,0.75,54,2,10.8,6.8,"Sydney Cricket Ground, Sydney",2011,Brisbane Heat,Sydney Sixers,Brisbane Heat,0,103.865854,43.682927,0.134146,147.682927,0.270292,111.648148,3.925926,115.833333,3.416667,158.975113,Sydney Sixers,Brisbane Heat,1500.0,1500.0,0.0,1500.0,1500.0,0.0,520,117.866667
3,524915,20,139,8,0,6.950000,15.444444,15.444444,1.00,35,3,7.0,1.0,"Sydney Cricket Ground, Sydney",2011,Brisbane Heat,Sydney Sixers,Brisbane Heat,0,152.366197,0.000000,-13.366197,139.000000,0.000000,160.019608,6.490196,162.416667,6.666667,158.975113,Sydney Sixers,Brisbane Heat,1500.0,1500.0,0.0,1500.0,1500.0,0.0,1112,15.444444
4,524916,6,37,1,14,6.166667,18.500000,1.233333,0.30,34,1,6.8,4.8,Melbourne Cricket Ground,2011,Sydney Thunder,Sydney Thunder,Melbourne Stars,0,47.295238,119.838095,-10.295238,156.838095,0.741510,45.593750,1.343750,46.333333,0.666667,160.144996,Melbourne Stars,Sydney Thunder,1500.0,1500.0,0.0,1500.0,1500.0,0.0,37,88.800000


## Step 3: verify the target column

In this step, we confirm that:
- `target_batfirst` is the true binary modelling target
- `winner` is only the winning team name, not the target itself
- `target_batfirst` matches the logic:
  
  `winner == batting_first_team`

In [17]:
# Step 3: verify target properly

print("target_batfirst unique values:")
print(sorted(df["target_batfirst"].dropna().unique()))

print("\ntarget_batfirst distribution:")
print(df["target_batfirst"].value_counts(dropna=False).sort_index())

print("\nSample values from winner:")
print(df["winner"].dropna().unique()[:10])

# Rebuild the binary target from winner and batting_first_team
target_check = (df["winner"] == df["batting_first_team"]).astype(int)

# Compare with stored target_batfirst
comparison = pd.DataFrame({
    "winner": df["winner"],
    "batting_first_team": df["batting_first_team"],
    "target_batfirst": df["target_batfirst"],
    "target_check": target_check
})

print("\nDo target_batfirst and reconstructed target match?")
print((comparison["target_batfirst"] == comparison["target_check"]).value_counts(dropna=False))

print("\nRows where they do NOT match:")
display(comparison[comparison["target_batfirst"] != comparison["target_check"]].head(10))

target_batfirst unique values:
[np.int64(0), np.int64(1)]

target_batfirst distribution:
target_batfirst
0    1260
1    1144
Name: count, dtype: int64

Sample values from winner:
['Sydney Sixers' 'Sydney Thunder' 'Adelaide Strikers' 'Hobart Hurricanes'
 'Melbourne Stars' 'Perth Scorchers' 'Melbourne Renegades' 'Brisbane Heat']

Do target_batfirst and reconstructed target match?
True    2404
Name: count, dtype: int64

Rows where they do NOT match:


,winner,batting_first_team,target_batfirst,target_check


## Step 4: define feature keep/drop decisions

In this step, we create an explicit feature audit table.

This helps document:
- which column is the modelling target
- which columns are dropped
- why each dropped column is excluded
- which columns are kept for modelling

This table will make the notebook easier to explain in the report and presentation.

In [18]:
# Step 4: define feature decisions

target_col = "target_batfirst"

drop_reasons = {
    "winner": "drop - winner team name, not a modelling feature; directly tied to outcome",
    "target_batfirst": "target - use as y, not as X",
    "runs": "drop - redundant with run_rate/checkpoint and part of correlated feature group",
    "runs_per_wicket": "drop - derived ratio, redundant with runs/wickets",
    "pressure_index": "drop - engineered from existing innings state, redundant/correlated",
    "checkpoint_ratio": "drop - directly determined by checkpoint",
    "momentum_runrate": "drop - derived from momentum_runs over balls/overs, redundant",
    "momentum_score": "drop - engineered combination, redundant with momentum features",
    "runs_x_wickets": "drop - interaction term, highly derived/redundant",
    "r_pw_x_momentum": "drop - interaction term, highly derived/redundant",
    "overs_left": "drop - perfectly determined by checkpoint",
    "exp_remain_final": "drop - future-information style expectation about remaining innings",
    "par_runs_cp_final": "drop - par/final expectation feature, potential leakage-style signal",
    "par_gap_cp": "drop - derived from par/final expectation features",
    "exp_final_20": "drop - uses expected final total information",
    "venue_mean_exp_final20": "drop - expectation of final total, redundant with other priors",
    "season": "drop - broad identifier, replaced by season aggregate priors",
    "team1": "drop - raw team identity, redundant with batting/toss/team setup",
    "team2": "drop - raw team identity, redundant with batting/toss/team setup",
    "elo_team1_pre": "drop - redundant with elo_diff_pre / elo_diff style features",
    "elo_team2_pre": "drop - redundant with elo_diff_pre / elo_diff style features",
    "elo_diff_pre": "drop - redundant with retained elo_diff",
    "elo_bat_first": "drop - redundant with retained elo_diff",
    "elo_bowl_first": "drop - redundant with retained elo_diff",
    "toss_winner": "drop - team identity style feature; excluded for cleaner setup",
    "batting_first_team": "drop - raw team identity tied to target construction",
}

kept_features = [
    "match_id",
    "checkpoint",
    "wickets",
    "run_rate",
    "momentum_runs",
    "momentum_wickets",
    "venue",
    "resource_frac",
    "venue_mean_runs_cp",
    "venue_mean_wkts_cp",
    "season_mean_runs_cp",
    "season_mean_wkts_cp",
    "elo_diff",
]

audit_rows = []

for col in df.columns:
    if col == target_col:
        status = "target"
        reason = drop_reasons[col]
    elif col in kept_features:
        status = "keep"
        reason = "keep - selected final modelling feature"
    elif col in drop_reasons:
        status = "drop"
        reason = drop_reasons[col]
    else:
        status = "review"
        reason = "not assigned yet"

    audit_rows.append({
        "column": col,
        "status": status,
        "reason": reason
    })

feature_audit_df = pd.DataFrame(audit_rows)

print("Feature audit summary:")
print(feature_audit_df["status"].value_counts())

print("\nKept features:")
print(feature_audit_df.loc[feature_audit_df["status"] == "keep", "column"].tolist())

print("\nTarget column:")
print(feature_audit_df.loc[feature_audit_df["status"] == "target", "column"].tolist())

print("\nAny unassigned columns?")
print(feature_audit_df.loc[feature_audit_df["status"] == "review", "column"].tolist())

display(feature_audit_df)

Feature audit summary:
status
drop      25
keep      13
target     1
Name: count, dtype: int64

Kept features:
['match_id', 'checkpoint', 'wickets', 'run_rate', 'momentum_runs', 'momentum_wickets', 'venue', 'resource_frac', 'venue_mean_runs_cp', 'venue_mean_wkts_cp', 'season_mean_runs_cp', 'season_mean_wkts_cp', 'elo_diff']

Target column:
['target_batfirst']

Any unassigned columns?
[]


,column,status,reason
0,match_id,keep,keep - selected final modelling feature
1,checkpoint,keep,keep - selected final modelling feature
2,runs,drop,drop - redundant with run_rate/checkpoint and ...
3,wickets,keep,keep - selected final modelling feature
4,overs_left,drop,drop - perfectly determined by checkpoint
5,run_rate,keep,keep - selected final modelling feature
6,runs_per_wicket,drop,"drop - derived ratio, redundant with runs/wickets"
7,pressure_index,drop,"drop - engineered from existing innings state,..."
8,checkpoint_ratio,drop,drop - directly determined by checkpoint
9,momentum_runs,keep,keep - selected final modelling feature


## Step 5: build cleaned modelling dataframe

Using the feature audit decisions above, we now create a clean modelling dataframe that contains:
- the final kept features
- the binary target column

This becomes the main dataset for checkpoint-specific modelling in Research Project B.

In [19]:
# Step 5: build cleaned modelling dataframe

model_df = df[kept_features + [target_col]].copy()

X = model_df[kept_features].copy()
y = model_df[target_col].copy()

print("model_df shape:", model_df.shape)
print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nmodel_df columns:")
print(model_df.columns.tolist())

print("\nMissing values in model_df:")
print(model_df.isna().sum()[model_df.isna().sum() > 0])

display(model_df.head())

model_df shape: (2404, 14)
X shape: (2404, 13)
y shape: (2404,)

model_df columns:
['match_id', 'checkpoint', 'wickets', 'run_rate', 'momentum_runs', 'momentum_wickets', 'venue', 'resource_frac', 'venue_mean_runs_cp', 'venue_mean_wkts_cp', 'season_mean_runs_cp', 'season_mean_wkts_cp', 'elo_diff', 'target_batfirst']

Missing values in model_df:
elo_diff    39
dtype: int64


,match_id,checkpoint,wickets,run_rate,momentum_runs,momentum_wickets,venue,resource_frac,venue_mean_runs_cp,venue_mean_wkts_cp,season_mean_runs_cp,season_mean_wkts_cp,elo_diff,target_batfirst
0,524915,6,0,6.166667,33,0,"Sydney Cricket Ground, Sydney",0.757142,45.982759,1.620690,46.333333,0.666667,0.0,0
1,524915,10,3,5.000000,22,3,"Sydney Cricket Ground, Sydney",0.523994,73.894737,2.649123,74.250000,1.666667,0.0,0
2,524915,15,5,6.933333,54,2,"Sydney Cricket Ground, Sydney",0.270292,111.648148,3.925926,115.833333,3.416667,0.0,0
3,524915,20,8,6.950000,35,3,"Sydney Cricket Ground, Sydney",0.000000,160.019608,6.490196,162.416667,6.666667,0.0,0
4,524916,6,1,6.166667,34,1,Melbourne Cricket Ground,0.741510,45.593750,1.343750,46.333333,0.666667,0.0,0


## Step 6: split into checkpoint-specific dataframes

We now split the cleaned modelling dataframe into four separate checkpoint datasets:
- 6 overs
- 10 overs
- 15 overs
- 20 overs

This is the base structure for training separate models for each checkpoint, as suggested in the supervisor feedback.

In [20]:
# Step 6: split into checkpoint-specific dataframes

df_6 = model_df[model_df["checkpoint"] == 6].copy()
df_10 = model_df[model_df["checkpoint"] == 10].copy()
df_15 = model_df[model_df["checkpoint"] == 15].copy()
df_20 = model_df[model_df["checkpoint"] == 20].copy()

checkpoint_dfs = {
    6: df_6,
    10: df_10,
    15: df_15,
    20: df_20,
}

for cp, d in checkpoint_dfs.items():
    print(f"Checkpoint {cp}")
    print("Shape:", d.shape)
    print("Unique checkpoint values:", d["checkpoint"].unique())
    print("-" * 40)

Checkpoint 6
Shape: (618, 14)
Unique checkpoint values: [6]
----------------------------------------
Checkpoint 10
Shape: (613, 14)
Unique checkpoint values: [10]
----------------------------------------
Checkpoint 15
Shape: (601, 14)
Unique checkpoint values: [15]
----------------------------------------
Checkpoint 20
Shape: (572, 14)
Unique checkpoint values: [20]
----------------------------------------


## Step 7: check class balance by checkpoint

We now inspect the target distribution separately for each checkpoint dataset.

This helps us:
- confirm that binary classification is valid at every checkpoint
- see whether class imbalance differs across checkpoints
- document the modelling setup clearly before training checkpoint-specific models

In [21]:
# Step 7: class balance for each checkpoint

for cp, d in checkpoint_dfs.items():
    print(f"Checkpoint {cp}")
    print("Target counts:")
    print(d["target_batfirst"].value_counts(dropna=False).sort_index())
    print("\nTarget proportions:")
    print(d["target_batfirst"].value_counts(normalize=True, dropna=False).sort_index().round(3))
    print("-" * 40)

Checkpoint 6
Target counts:
target_batfirst
0    327
1    291
Name: count, dtype: int64

Target proportions:
target_batfirst
0    0.529
1    0.471
Name: proportion, dtype: float64
----------------------------------------
Checkpoint 10
Target counts:
target_batfirst
0    323
1    290
Name: count, dtype: int64

Target proportions:
target_batfirst
0    0.527
1    0.473
Name: proportion, dtype: float64
----------------------------------------
Checkpoint 15
Target counts:
target_batfirst
0    315
1    286
Name: count, dtype: int64

Target proportions:
target_batfirst
0    0.524
1    0.476
Name: proportion, dtype: float64
----------------------------------------
Checkpoint 20
Target counts:
target_batfirst
0    295
1    277
Name: count, dtype: int64

Target proportions:
target_batfirst
0    0.516
1    0.484
Name: proportion, dtype: float64
----------------------------------------


## Audit summary

Key findings from this data audit:

- The final modelling dataset contains **2404 rows** and **39 columns**
- The correct binary target is **`target_batfirst`**
- The column **`winner`** is **not** the target; it contains winning team names
- We verified that `target_batfirst` exactly matches the logic:

  `winner == batting_first_team`

- Based on leakage, redundancy, and correlation review:
  - **25 columns** were dropped
  - **13 columns** were retained as final modelling features
- The cleaned modelling dataframe has shape **(2404, 14)**:
  - **13 features**
  - **1 target**
- The only remaining missing values in the cleaned modelling dataframe are in **`elo_diff`** (**39 rows**)
- The cleaned modelling dataframe was split into checkpoint-specific datasets:
  - **df_6:** 618 rows
  - **df_10:** 613 rows
  - **df_15:** 601 rows
  - **df_20:** 572 rows
- Class balance is reasonably stable across all checkpoints, with both classes close to evenly distributed

Conclusion:
This notebook establishes a clean and reproducible checkpoint-based modelling dataset for Research Project B. The next step is to create a separate modelling notebook and train a **CatBoost classifier for one checkpoint first**, before extending to all four checkpoints.